In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'WenQuanYi Micro Hei']
plt.rcParams['axes.unicode_minus'] = False

In [2]:
# 读取噪声特征数据
df_noise = pd.read_csv(r'../数据/噪声特征.csv', encoding='utf-8-sig')
print("原始数据:")
print(df_noise.head())

原始数据:
        code  收益率标准差（波动率）         偏度           峰度   一阶自相关系数   五阶自相关系数  \
0  688256.SH     0.108846  28.037147   928.267001  0.001087  0.006353   
1  600522.SH     0.097588 -41.675543  1921.391796 -0.029507  0.005646   
2  688126.SH     0.096747 -30.538870  1067.504167 -0.042972 -0.242171   
3  603993.SH     0.096052 -42.773695  2004.706994 -0.034636  0.015739   
4  300316.SZ     0.091850 -39.936163  1830.945311 -0.030272 -0.024503   

   波动率聚集变异系数     极端值比例      噪声信号比  
0   0.704416  0.001523  28.292492  
1   1.056130  0.000827  60.314735  
2   0.924229  0.002180  50.629059  
3   0.847417  0.000416  91.484115  
4   0.749028  0.001250  97.552593  


In [3]:
from sklearn.preprocessing import MinMaxScaler

# 选择用于分组的指标（排除异常值影响）
# 对峰度进行对数变换，降低极端值影响
df_noise['log_kurtosis'] = np.log1p(df_noise['峰度'])

# 标准化
scaler = MinMaxScaler()
df_noise['std_scaled'] = scaler.fit_transform(df_noise[['收益率标准差（波动率）']])
df_noise['kurt_scaled'] = scaler.fit_transform(df_noise[['log_kurtosis']])
df_noise['autocorr_scaled'] = scaler.fit_transform(df_noise[['一阶自相关系数']])
df_noise['extreme_scaled'] = scaler.fit_transform(df_noise[['极端值比例']])
df_noise['vol_cluster_scaled'] = scaler.fit_transform(df_noise[['波动率聚集变异系数']])

# 综合噪声得分（权重分配）
# 波动率40%、峰度30%、极端值20%、波动聚集10%
df_noise['noise_score'] = (0.40 * df_noise['std_scaled'] + 
                           0.30 * df_noise['kurt_scaled'] + 
                           0.20 * df_noise['extreme_scaled'] +
                           0.10 * df_noise['vol_cluster_scaled'])

# 按噪声得分分组
df_noise = df_noise.sort_values('noise_score', ascending=False).reset_index(drop=True)

# 分三组（自动三等分）
n_stocks = len(df_noise)

# 自动三等分（向上取整，保证每组数量均匀）
group1 = int(np.ceil(n_stocks / 3))  # 高噪声组
group2 = int(np.ceil(n_stocks * 2 / 3))  # 中噪声组结束位置

df_noise['group'] = '中噪声组'
df_noise.loc[:group1-1, 'group'] = '高噪声组'   # 前1/3
df_noise.loc[group2:, 'group'] = '低噪声组'    # 后1/3

print("\n=== 分组结果 ===")
for group in ['高噪声组', '中噪声组', '低噪声组']:
    stocks = df_noise[df_noise['group'] == group]['code'].tolist()
    print(f"\n{group} ({len(stocks)}只):")
    print(stocks)
    print(f"平均噪声得分: {df_noise[df_noise['group'] == group]['noise_score'].mean():.3f}")

# 保存分组结果
df_noise[['code', 'group', 'noise_score', '收益率标准差（波动率）', 
          '峰度', '一阶自相关系数']].to_csv(r'../数据/噪声分组.csv', index=False)


=== 分组结果 ===

高噪声组 (100只):
['600522.SH', '688256.SH', '603993.SH', '688126.SH', '300316.SZ', '600519.SH', '300759.SZ', '688169.SH', '003816.SZ', '601136.SH', '688009.SH', '000425.SZ', '002384.SZ', '600930.SH', '600795.SH', '688012.SH', '601698.SH', '001391.SZ', '601319.SH', '601111.SH', '600905.SH', '002463.SZ', '601898.SH', '300803.SZ', '601728.SH', '688271.SH', '300408.SZ', '300059.SZ', '002600.SZ', '000617.SZ', '300308.SZ', '600346.SH', '000338.SZ', '600010.SH', '601995.SH', '600039.SH', '688223.SH', '600438.SH', '002050.SZ', '600115.SH', '601058.SH', '688036.SH', '000100.SZ', '000708.SZ', '601021.SH', '688472.SH', '600674.SH', '002304.SZ', '002920.SZ', '000876.SZ', '603019.SH', '000983.SZ', '603799.SH', '601998.SH', '000725.SZ', '300033.SZ', '601360.SH', '300896.SZ', '002648.SZ', '688506.SH', '600188.SH', '300498.SZ', '601868.SH', '300251.SZ', '601901.SH', '000792.SZ', '600886.SH', '601628.SH', '300014.SZ', '002601.SZ', '600219.SH', '600900.SH', '603893.SH', '603986.SH', '600000.S

In [4]:
# 根据股票的噪声特征设计滤波器参数
def design_filter_params(row, group_mean_std):
    """
    返回:
    - cutoff_freq: 截止频率 (归一化频率，0-0.5)
    - filter_order: 滤波器阶数
    - filter_type: 滤波器类型
    """
    std = row['收益率标准差（波动率）']
    kurt = row['峰度']
    autocorr = row['一阶自相关系数']
    extreme = row['极端值比例']
    group = row['group']
    
    # ---------------------- 核心：自动获取本组平均波动率 ----------------------
    avg_std = group_mean_std[group]  # 自动从分组统计中取

    # 1. 截止频率计算（自适应周期）
    if group == '高噪声组':
        base_period = 40
    elif group == '低噪声组':
        base_period = 20
    else:
        base_period = 30

    # 自适应调整周期：当前波动率 / 本组平均波动率
    period_adj = base_period * (std / avg_std)
    
    # 限制周期范围
    period = np.clip(period_adj, 15, 60)
    cutoff_freq = 1.0 / period  # 归一化截止频率
    
    # 2. 滤波器阶数
    if kurt > 10:
        order = 6
    elif kurt > 5:
        order = 5
    elif kurt > 3:
        order = 4
    else:
        order = 3
    
    # 自相关高 → 降低阶数
    if autocorr > 0.1:
        order = max(3, order - 1)
    
    # 3. 滤波器类型
    if extreme > 0.02:
        filter_type = 'butterworth'
    else:
        filter_type = 'chebyshev'
    
    return {
        'cutoff_freq': cutoff_freq,
        'cutoff_period': period,
        'filter_order': order,
        'filter_type': filter_type
    }

# ===================== 【自动计算每组平均波动率】=====================
group_mean_std = df_noise.groupby('group')['收益率标准差（波动率）'].mean().to_dict()

print("=== 自动计算的各组平均波动率（自适应基准）===")
for g, v in group_mean_std.items():
    print(f"{g} : {v:.4f}")

# ===================== 为每只股票生成滤波器参数 =====================
filter_params = []
for _, row in df_noise.iterrows():
    params = design_filter_params(row, group_mean_std)  # 传入自动计算的均值
    params['code'] = row['code']
    params['group'] = row['group']
    filter_params.append(params)

filter_df = pd.DataFrame(filter_params)

# ===================== 输出 & 保存 =====================
print("\n=== 滤波器参数设计结果 ===")
print(filter_df.to_string(index=False))

filter_df.to_csv(r'../数据/滤波器参数设置.csv', index=False, encoding='utf-8-sig')

=== 自动计算的各组平均波动率（自适应基准）===
中噪声组 : 0.0331
低噪声组 : 0.0260
高噪声组 : 0.0553

=== 滤波器参数设计结果 ===
 cutoff_freq  cutoff_period  filter_order filter_type      code group
    0.016667      60.000000             6   chebyshev 600522.SH  高噪声组
    0.016667      60.000000             6   chebyshev 688256.SH  高噪声组
    0.016667      60.000000             6   chebyshev 603993.SH  高噪声组
    0.016667      60.000000             6   chebyshev 688126.SH  高噪声组
    0.016667      60.000000             6   chebyshev 300316.SZ  高噪声组
    0.016869      59.278739             6   chebyshev 600519.SH  高噪声组
    0.016667      60.000000             6   chebyshev 300759.SZ  高噪声组
    0.016667      60.000000             6   chebyshev 688169.SH  高噪声组
    0.019718      50.715810             6   chebyshev 003816.SZ  高噪声组
    0.016667      60.000000             6   chebyshev 601136.SH  高噪声组
    0.019228      52.008071             6   chebyshev 688009.SH  高噪声组
    0.019237      51.983597             6   chebyshev 000425.SZ  高噪声组
  